# Debugging ed eccezioni

Una delle attività più importanti per i programmatori è il cosiddetto **debugging**. Si tratta del processo attraverso cui si cerca di capire dove si trovino gli errori (bug) nel nostro codice. A seconda della complessità del codice, può essere un processo tedioso e per questo vale la pena di dedicare una lezione ad alcuni aspetti legati al debugging che non abbiamo discusso in altri giorni.

Partiamo quindi con la discussione di un aspetto del linguaggio Python che abbiamo visto (senza discuterlo) e che ci serve per intrudurre altri concetti legati al debug.

## Eccezioni (exception) e traceback 

Le **eccezioni** (*exception* in inglese) sono un meccanismo dei linguaggi di programmazione che viene usato per segnalare quando qualcosa è andato storto. Nel caso di Python ci viene anche segnalato il punto esatto del codice in cui si è verificata l'eccezione, nonché che tipo di eccezione si è verificata. Partiamo dall'esempio più semplice possibile: cosa succede se dividiamo un numero per 0?

In [ ]:
def divisione(x, y):
    return x/y

divisione(10,0)

Osservando l'output vediamo che Python, dopo aver prodotto una serie di `-` ci segnala nella prima riga che è avvenuta un'eccezione (in questo caso un `ZeroDivisionError`). Sulla destra di `ZeroDivisionError` Python scrive `Traceback (most recent call last)`, indicandoci che nelle righe seguenti apparirà un traceback, ma di cosa si tratta?

Un **traceback** o **stack trace** è una sequenza di informazioni che ci dice esattamente in che parte del codice è avvenuto l'errore. Nel caso del nostro esempio, vediamo che ci sono due sequenze di tre righe (che cominciano con `Cell`). Analizziamo la prima riga di ciascun blocco:
```
Cell In[1], line 4
[...]
Cell In[1], line 2, in divisione(x, y)
```
La prima riga ci indica che l'errore è avvenuto in una cella, alla riga 4. Cell In\[1\] indica che l'errore è avvenuto nella prima cella che abbiamo eseguito (purtroppo se la rieseguiamo questo numero aumenta e non ci viene detto in che cella specifica sia il codice. Nel secondo blocco troviamo l'errore vero e proprio, che è alla riga 2 nella funzione `divisione(x,y)`

Sotto alle espressioni `Cell In...` Python ci mostra alcune righe di codice che precedono quella in cui è avvenuto l'errore. Oltre alle righe viene evidenziato il punto preciso in cui si è verificato l'errore.

E se invece che una chiamata a funzione ce ne fossero due?

In [ ]:
def divisione2(x, y):
    return divisione(x,y)

divisione2(1,0)

Come vedete viene aumentato il numero di funzioni che vengono visualizzate nel traceback! È per questo che il traceback è anche chiamato stack trace: viene visualizzata una **pila** (*stack*) in cui sono "impilate" tutte le invocazioni di funzioni e metodi che hanno portato all'errore. Possiamo quindi ricostruire la catena di eventi che ha portato alla divisione per 0. Notate anche come l'ultimo blocco, relativo alla funzione `divisione(x, y)` menzioni `Cell In[1]`, un numero di cella diverso da `divisione2`!

E se l'errore avvenisse in un altro modulo cosa succederebbe? Ho implementato nel modulo `src/divisione.py` una funzione `divisione_modulo(x, y)` che fa una divisione fra due numeri, vediamo che succede se la invoco da un notebook?

In [ ]:
from src.divisione import divisione_modulo

divisione_modulo(1,0)

Come vedete l'espressione è cambiata! Il primo blocco contiene sempre `Cell` all'inizio, mentre il secondo ci indica esattamente in quale file è avvenuto l'errore! Lo stesso avviene se invochiamo in modo errato la funzione `open`

In [ ]:
with open("nonesiste.txt") as f:
    pass

In questo caso otteniamo un'eccezione `FileNotFoundError` quando viene chiamata la funzione `io_open`. In questo caso non vediamo il codice di `io_open` perché si tratta di una funzione di libreria che non è direttamente scritta in Python.

E se invece usassimo una funzione di libreria? Vediamo che succede usando interi al posto di stringhe

In [ ]:
import re

re.findall(1,2)

Come vedete l'errore avviene dentro alla libreria `re` nel file `__init__.py`! L'eccezione che viene creata è `TypeError`, una delle più comuni. Indica che il tipo di file che abbiamo usato come primo argomento non è corretto. NB: il testo ci dice anche che è il primo argomento che è errato (nell'ultima riga del traceback).

E se usassimo una libreria di terze parti? Vediamo che succede!

In [ ]:
import pandas as pd

pd.DataFrame("ciao")

In questo caso otteniamo un'eccezione di tipo `ValueError` (errore nel valore di un argomento) nel file `core/frame.py` della libreria `pandas`. Nel caso di errori che coinvolgono librerie esterne, dobbiamo tipicamente controllare l'ultima parte del nostro codice nel tracebak, in modo da individuare la riga di codice che causa l'errore.

## Lanciare eccezioni nel nostro codice

E se volessimo lanciare un'eccezione noi? Come potremmo fare? Vediamo ad esempio il caso della funzione ricorsiva fattoriale che abbiamo visto ieri:

In [ ]:
def fattoriale(n):
    if not isinstance(n, int):
        raise TypeError("il fattoriale è definito solo per numeri interi")
        
    if n < 0:
        raise ValueError("il fattoriale non ha senso per un numero minore di 0")

    if n == 1 or n == 0:
        return 1

    return n * fattoriale(n - 1)

fattoriale(1)

In [ ]:
fattoriale(-10)

In [ ]:
fattoriale(0.1)

Si possono anche definire eccezioni personalizzate, ma oggi non ne parleremo (se vi interessa, google vi può aiutare!).

## Try/except

Come vedete possiamo gestire vari tipi di errori e fornire una descrizione per il tipo di errore che abbiamo causato. Tuttavia, ogni volta che viene lanciata un'eccezione il nostro codice va in crash! E se volessimo gestire questo errore in altro modo? Potremmo utilizzare la sintassi `try/except`, che ci consente di "catturare" le eccezioni ed evitare che crashi l'intera cella. Vediamo come:

In [ ]:
try:
    fattoriale(-10)
except ValueError as e:
    print("oops")
    print(e)
    print("l'esecuzione continua!")

Inseriamo una o più righe di codice in `try` e poi specifichiamo il tipo di eccezione che vogliamo catturare. Python ci consente anche di gestire tutti i tipi di eccezione senza specificare, ma questo è formtemente sconsigliato e io non vi mostrerò come farlo. È invece bene capire che tipi di eccezzioni verranno sollevate dal codice e gestirle manualmente.

Come avete notato nella cella precedente abbiamo stampato il contenuto dell'eccezione. E se invece volessimo visualizzare l'intero traceback? Per farlo dobbiamo usare una libreria di sistema chiamata `traceback` e il suo metodo `.format_exc()`

In [ ]:
import traceback

try:
    fattoriale(-10)
except ValueError as e:
    print("oops")
    print(traceback.format_exc())
    print("l'esecuzione continua!")

## Consigli per il debugging

Oltre a situazioni in cui il programma (o la cella) crasha completamente, ci sono molti casi in cui il programma continua ad eseguire ma non ci fornisce i risultati attesi per qualche caso. In questo caso possiamo procedere per gradi, stampando il risultato di ciascuna espressione. Vediamo un semplice esempio: una funzione che deve determinare se una stringa è palindroma.

In [ ]:
def is_palindromo(parola):
    return parola == parola[::-1]

print(is_palindromo("Otto"))  # Dovrebbe restituire True

A un primo sguardo il codice sembrerebbe giusto. Eppure il nostro codice non funziona! Investighiamo.

In [ ]:
def is_palindromo(parola):
    print(f"stiamo comparando {parola} con {parola[::-1]}")
    return parola == parola[::-1]

print(is_palindromo("Otto"))# Dovrebbe restituire True

Stampando ci rendiamo conto che manca un `.lower()`. Ora possiamo risolvere il problema:

In [ ]:
def is_palindromo(parola):
    parola = parola.lower()
    print(f"stiamo comparando {parola} con {parola[::-1]}")
    return parola == parola[::-1]

print(is_palindromo("Otto"))  # Dovrebbe restituire True

# Esercizi

1. Questa funzione dovrebbe contare le vocali in una stringa, ma crasha. Perché il codice crasha? Come lo sistemeresti? C'è anche un errore logico...

In [ ]:
 # ES1

def conta_vocali(testo):
    vocali = 'aeoui'
    count = 1
    for char in testo:
        if char in volwels:
            count += 1
    return count

print(conta_vocali("linguistica"))  # Dovrebbe stampare 5

2. La funzione `normalize` dovrebbe aprire un file e restituire tutto il suo contenuto in minuscolo. Tuttavia, la funzione non gestisce il caso in cui il file non esista. Gestisci l'eccezione appropriata e stampa un errore. 

In [ ]:

def normalize(nome_file):
    with open(nome_file) as f:
        text = f.read()
    text = text.lower()

stampa_punti("nonesiste.txt")

3. Questo codice dovrebbe rimuovere tutta la punteggiatura da una stringa e renderla minuscola, tuttavia crasha. Risolvi il problema.

In [ ]:
import string

def pulisci_testo(testo):
    for p in string.punctuation:
        testo = testo.replace(p, "") 
    return testo.lowercase()

print(pulisci_testo("Ciao, mondo!"))  # Dovrebbe restituire "ciao mondo"

4. La funzione `media_lista` dovrebbe calcolare il valore medio di una lista. Tuttavia, crasha quando la usiamo su una lista vuota. Risolvi il problema usando `try/catch` all'interno della funzione. Se la lista è vuota deve ritornare `None`.

In [ ]:
def media_lista(l):
    return sum(l)/len(l)

media_lista([])

5. La funzione `lista_fibonacci` dovrebbe restituire una lista di tutti i numeri nella sequenza di Fibonacci dall'x-esimo al y-esimo. Tuttavia non funziona e contiene vari bug. Risolvi i bug.

In [ ]:
def fibonacci(n):
    if n == 0:
        return 1
    return fibonacci(n -1) + fibonacci(n - 2)
        

def lista_fibonacci(x, y):
    lista_fibonacci = []
    for i in range(x, y):
        lista_fibonacci[i] = fibonacci(i)
    return fibonacci

lista_fibonacci(0,10)

6. La funzione seguente dovrebbe contare il numero di volte in cui appare un numero in una lista. Tuttavia, ha vari bug. Sistema i problemi.

In [ ]:
def conta_interi(lista):
    dizionario = {}
    i = 0
    while i < len(lista):
        dizionario[i] += 1
    return dizionario

lista_interi = [8, 6, 16, 3, 16, 6, 17, 17, 6, 5]
conta_interi(lista_interi)